Import

In [14]:
import numpy as np
import pandas as pd
import random


Data - SLP

In [55]:
# Standard-Lastprofil für 2019 (stündlich)
file_path = r"D:\Dokumente\Studium\20242025 WS\WattWise\Repo\wattWise\data\slp\GGV_SLP_1000_MWh_2019_01.xlsx"
sheet_name = "15Min - SLP 2019" # Single Load Profile 2019

df_standard_load_profile_2019 = pd.read_excel(file_path, sheet_name=sheet_name)
df_standard_load_profile_2019



,Messwert-Nr.,Wochentag,Datum,Zeitstempel von,Zeitstempel bis,MP von,MP bis,H00 [kW],H00 [kWh]
0,1,2,2019-01-01,2019-01-01 00:00:00.000,2019-01-01 00:15:00.000,00:00,00:15,145.200,36.300
1,2,2,2019-01-01,2019-01-01 00:15:00.000,2019-01-01 00:30:00.000,00:15,00:30,140.112,35.028
2,3,2,2019-01-01,2019-01-01 00:30:00.000,2019-01-01 00:45:00.000,00:30,00:45,135.720,33.930
3,4,2,2019-01-01,2019-01-01 00:45:00.000,2019-01-01 01:00:00.000,00:45,01:00,130.732,32.683
4,5,2,2019-01-01,2019-01-01 01:00:00.000,2019-01-01 01:15:00.000,01:00,01:15,127.236,31.809
...,...,...,...,...,...,...,...,...,...
35035,35036,2,2019-12-31,2019-12-31 22:44:59.999,2019-12-31 22:59:59.999,22:45,23:00,158.172,39.543
35036,35037,2,2019-12-31,2019-12-31 22:59:59.999,2019-12-31 23:14:59.999,23:00,23:15,158.672,39.668
35037,35038,2,2019-12-31,2019-12-31 23:14:59.999,2019-12-31 23:29:59.999,23:15,23:30,155.480,38.870
35038,35039,2,2019-12-31,2019-12-31 23:29:59.999,2019-12-31 23:44:59.999,23:30,23:45,151.888,37.972


In [56]:
def fill_missing_hours(df: pd.DataFrame, time_col: str = "Time", value_cols: list = None,
                       year: int = 2019, fill_value=None) -> pd.DataFrame:
    """
    Erkennt fehlende Stunden in einem Zeitreihen-DataFrame und füllt sie optional mit fill_value (z. B. 0 oder NaN).

    Args:
        df (DataFrame): Zeitreihen-DataFrame mit einer Zeitspalte.
        time_col (str): Name der Spalte mit Timestamps.
        value_cols (list): Spalten, die beibehalten/gefüllt werden sollen. Wenn None → alle außer time_col.
        year (int): Jahr, für das die Stunden geprüft werden sollen.
        fill_value: Wert zum Auffüllen (z. B. 0, np.nan oder dict).

    Returns:
        DataFrame mit allen 8760 Stunden (oder 8784 bei Schaltjahr), vollständig aufgefüllt.
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df.set_index(time_col, inplace=True)

    # Generiere kompletten Stundenindex für das Jahr
    start = f"{year}-01-01 00:00"
    end = f"{year}-12-31 23:00"
    full_index = pd.date_range(start=start, end=end, freq="H")

    # Bestimme Spalten
    if value_cols is None:
        value_cols = [col for col in df.columns if col != time_col]

    # Reindex & auffüllen
    df_full = df.reindex(full_index)

    if fill_value is not None:
        df_full[value_cols] = df_full[value_cols].fillna(fill_value)

    df_full.reset_index(inplace=True)
    df_full.rename(columns={"index": time_col}, inplace=True)

    return df_full

In [57]:
#Prepare DataFrame
df_standard_load_profile_2019.drop(columns=['Messwert-Nr.','MP von', 'MP bis'], inplace=True)

#Fasse Zeitstempel pro Stunde zusammen und summiere H00 [kW] in neuer df
df_standard_load_profile_2019['Time'] = df_standard_load_profile_2019['Zeitstempel von'].dt.floor('h')

df_agg = df_standard_load_profile_2019.groupby('Time').agg({'H00 [kW]': 'sum'}).reset_index()
df_agg['H00 [kWh]'] = df_agg['H00 [kW]'] / 4 

df_standard_load_profile_2019 = fill_missing_hours(df=df_agg, time_col='Time', value_cols=['H00 [kWh]'],
    year=2019,
    fill_value=0 # oder np.nan
)

df_standard_load_profile_2019

C:\Users\Hannah Heimpel\AppData\Local\Temp\ipykernel_18872\386875733.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=start, end=end, freq="H")


,Time,H00 [kW],H00 [kWh]
0,2019-01-01 00:00:00,551.764,137.941
1,2019-01-01 01:00:00,492.584,123.146
2,2019-01-01 02:00:00,448.376,112.094
3,2019-01-01 03:00:00,418.236,104.559
4,2019-01-01 04:00:00,386.208,96.552
...,...,...,...
8755,2019-12-31 19:00:00,656.444,164.111
8756,2019-12-31 20:00:00,574.216,143.554
8757,2019-12-31 21:00:00,557.048,139.262
8758,2019-12-31 22:00:00,640.876,160.219


In [58]:
#add 'Wochentag'
df_standard_load_profile_2019['Weekday'] = df_standard_load_profile_2019['Time'].dt.day_name()

#add Arbeitstag oder Wochenende als neue Spalte
df_standard_load_profile_2019['Weekend'] = df_standard_load_profile_2019['Weekday'].apply(lambda x: 1 if x in ['Saturday', 'Sunday'] else 0)

#add Hour and save as int
df_standard_load_profile_2019['Hour'] = df_standard_load_profile_2019['Time'].dt.time
df_standard_load_profile_2019['Hour'] = df_standard_load_profile_2019['Hour'].apply(lambda x: x.hour)

#Normalizing the values to a "average household" with 3000 kWh  
df_standard_load_profile_2019['Normalized Load'] = df_standard_load_profile_2019['H00 [kWh]'] /df_standard_load_profile_2019['H00 [kWh]'].sum()*3000

#Save an seaonality index for each hour on a annual mean of 1.0
df_standard_load_profile_2019['Seasonal Factor'] = df_standard_load_profile_2019['Normalized Load'] / df_standard_load_profile_2019['Normalized Load'].mean()

Data - 2 week customer data

_1. Input_ zwei Wochen Customer Data 

In [41]:
# 2 Wochen Kundenverbrauch für 2020 (stündlich) -- hier ggf. Input ändern #TODO
df_random_two_week_customer_load_2020 = pd.DataFrame(np.random.rand(24 * 14) * 2, columns=['Load'])

#add random dates for 2 weeks in 2020 on hourly basis -- hier gfg. Input ändern #TODO
start_date = pd.to_datetime('2020-01-01 00:00:00')
end_date = pd.to_datetime('2020-01-14 23:00:00')
# Create a date range with hourly frequency
date_range = pd.date_range(start=start_date, end=end_date, freq='h')
df_random_two_week_customer_load_2020['Time'] = date_range

# Jahresverbrauch des Kunden in kWh
customer_annual_consumption_preyear = 3000 # kWh #TODO hier ggf. Input ändern

df_random_two_week_customer_load_2020

,Load,Time
0,0.113116,2020-01-01 00:00:00
1,1.422135,2020-01-01 01:00:00
2,1.291988,2020-01-01 02:00:00
3,0.946727,2020-01-01 03:00:00
4,1.324431,2020-01-01 04:00:00
...,...,...
331,1.134321,2020-01-14 19:00:00
332,1.082060,2020-01-14 20:00:00
333,1.283545,2020-01-14 21:00:00
334,1.665581,2020-01-14 22:00:00


In [43]:
# Funktion, die aus einem DataFrame mit Zeitstempeln und Lastwerten Dictionary erzeugt, das Wochentage als Keys und Listen von Tagesprofilen (DataFrames mit 24 Stunden) als Values enthält.
def build_weekday_profile_dict(df: pd.DataFrame, timestamp_col: str = 'Time', load_col: str = 'Load') -> dict:
    """
    Baut ein Dictionary mit Wochentags-Keys (Monday–Sunday) und Listen von Tagesprofilen (DataFrames mit 24 Stunden) als Values.

    Args:
        df (pd.DataFrame): DataFrame mit Zeitreihe über mehrere Tage (14 Tage). Muss Spalten für Zeitstempel und Last enthalten.
        timestamp_col (str): Name der Zeitstempelspalte. Default: 'Time'.
        load_col (str): Name der Lastspalte. Default: 'Load'.

    Returns:
        dict: Dictionary mit Wochentagen als Keys und Listen von Tages-DataFrames (je 24 Stunden) als Values.
    """

    # Stelle sicher, dass Time im richtigen Format ist
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df['Weekday'] = df[timestamp_col].dt.day_name()
    df['Date'] = df[timestamp_col].dt.date

    # Nach Tag gruppieren
    grouped_by_day = df.groupby('Date')

    # Dictionary zum Befüllen
    weekday_profiles = {}

    for date, group in grouped_by_day:
        if len(group) != 24:
            continue  # überspringe unvollständige Tage

        weekday = group['Weekday'].iloc[0]

        if weekday not in weekday_profiles:
            weekday_profiles[weekday] = []

        # Nur die Lastwerte als DataFrame (Index 0–23)
        daily_profile = group[[load_col]].reset_index(drop=True)
        weekday_profiles[weekday].append(daily_profile)

    return weekday_profiles




Jahreslastprofilerstellung

1. Random Jahresprofil erstellen

In [44]:
#Pro Wochentag wähle ein Lastprofi aus two week data zufällig aus
def generate_random_yearly_profile(weekday_dfs_with_load: dict, year: int) -> pd.DataFrame:
    """
    Erstellt ein zufälliges Jahreslastprofil, indem für jeden Tag im Jahr ein zufälliges Tagesprofil
    aus dem passenden Wochentags-Pool aus `weekday_dfs_with_load` gezogen wird.

    Args:
        weekday_dfs_with_load (dict): Dictionary mit Wochentagsnamen als Keys und Listen von Tagesprofilen (DataFrames mit 24 Stunden) als Values.
        year (int): Jahr, für das das Lastprofil erzeugt werden soll.

    Returns:
        pd.DataFrame: Zeitreihe mit stündlichem Lastprofil für das gesamte Jahr.
    """

    # Erstelle Zeitindex für das ganze Jahr (stündlich)
    start = pd.Timestamp(f"{year}-01-01 00:00")
    end = pd.Timestamp(f"{year}-12-31 23:00")
    full_index = pd.date_range(start, end, freq='h')

    # Leerer DataFrame für das Jahresprofil
    full_year_profile = pd.DataFrame(index=full_index)
    full_year_profile['Load'] = 0.0

    # Gruppiere den Index nach Datum
    grouped = full_year_profile.groupby(full_year_profile.index.date)

    for day, group in grouped:
        weekday_name = pd.Timestamp(day).day_name() #Wochentag des Tages

        # Ziehe zufälliges Tagesprofil aus passendem Pool
        profile_list = weekday_dfs_with_load.get(weekday_name, [])
        
        chosen_profile = random.choice(profile_list)

        # Trage die Lastwerte in die passenden Stunden ein
        full_year_profile.loc[group.index, 'Load'] = chosen_profile['Load'].values

    return full_year_profile
    

2. Monte Carlo Simulation über mehrere Lastprofile

In [45]:
#Funktion Monte Carlo Simulation über mehrere Lastprofile. Rufe dafür generate_random_yearly_profile über das selbe Jahr
def monte_carlo_simulation_load_profiles(weekday_dfs_with_load: dict, year: int, n_simulations: int) -> list:
    """
    Führt eine Monte-Carlo-Simulation durch, bei der mehrere zufällige Jahreslastprofile generiert werden.

    Args:
        weekday_dfs_with_load (dict): Dictionary mit Wochentagsnamen als Keys und Listen von Tagesprofilen (DataFrames mit 24 Stunden) als Values.
        year (int): Jahr, für das die Profile generiert werden.
        n_simulations (int): Anzahl der gewünschten Simulationen.

    Returns:
        list: Liste mit DataFrames, jeweils ein vollständiges Jahreslastprofil.
    """
    simulations = []

    for i in range(n_simulations):
        simulated_profile = generate_random_yearly_profile(weekday_dfs_with_load, year)
        simulations.append(simulated_profile)

    return simulations


3. Seasonality

In [102]:
# Jedem MOnat/Kalenderweoche aus SLP ein Seasonality Factor hinzufügen
def calculate_seasonal_factor_per_x(df_slp: pd.DataFrame, x: str = "Month"):
    """
    Berechnet den saisonalen Faktor für jeden Monat oder Tag im DataFrame.

    Args:
        df_slp (DataFrame): DataFrame Standardlastprofil mit einer Zeitspalte und einer Spalte für den saisonalen Faktor.
        x (str): 'Month' oder 'Day', 'Calendar Week' um den saisonalen Faktor zu berechnen.

    Returns:
        DataFrame mit dem saisonalen Faktor.
    """
    if x == "Month":
        seasonal_factor = df_slp.groupby(df_slp['Time'].dt.month)['Seasonal Factor'].mean().reset_index()
        seasonal_factor.rename(columns={'Seasonal Factor': 'Seasonal Factor'}, inplace=True)
    elif x == "Day":
        seasonal_factor = df_slp.groupby(df_slp['Time'].dt.day)['Seasonal Factor'].mean().reset_index()
        seasonal_factor.rename(columns={'Seasonal Factor': 'Seasonal Factor'}, inplace=True)
    elif x == "Calendar Week":
        seasonal_factor = df_slp.groupby(df_slp['Time'].dt.isocalendar().week)['Seasonal Factor'].mean().reset_index()
        seasonal_factor.rename(columns={'Seasonal Factor': 'Seasonal Factor'}, inplace=True)
    else:
        raise ValueError("x must be either 'Month' or 'Day' or 'Calendar Week'.")

    return seasonal_factor

# Checken, aus welcher Jahreszeit two week data ist. #TODO weil das natürlich Auswirkungen auf miteinberechnete Seasonality hat? Oder wird das mathematisch durch Normalisierung am Ende egal betrachtet?


#Jahreslastprofile mit Seasonality Factor multiplizieren
def apply_seasonal_factors(simulations: list, seasonal_factors: pd.DataFrame, by: str = "Month") -> list:
    """
    Multipliziert jeden Load-Wert in den Simulationen mit dem passenden saisonalen Faktor
    entweder nach Monat oder nach Kalenderwoche.

    Args:
        simulations (list): Liste von DataFrames mit datetime-index und 'Load'-Spalte.
        seasonal_factors (pd.DataFrame): DataFrame mit Spalten 'Month' oder 'Calendar Week' und 'Seasonal Factor'.
        by (str): 'Month' oder 'Calendar Week', je nachdem wie die Faktoren aufgebaut sind.

    Returns:
        list: Liste angepasster DataFrames mit modifizierten 'Load'-Werten.
    """
    if by not in ['Month', 'Calendar Week']:
        raise ValueError("Parameter 'by' muss 'Month' oder 'Calendar Week' sein.")

    adjusted_simulations = []

    for df in simulations:
        df = df.copy()
        df['Time'] = pd.to_datetime(df.index)
        if by == 'Month':
            df['TimeKey'] = df.index.month
            merge_col = 'Time'
        elif by == 'Calendar Week':
            df['TimeKey'] = df.index.isocalendar().week
            merge_col = 'Time'

        # Merge mit saisonalen Faktoren
        df = df.merge(seasonal_factors, how='left', left_on='TimeKey', right_on=merge_col)
        df['Load'] = df['Load'] * df['Seasonal Factor']

        # Aufräumen
        df.drop(columns=['Time_y', 'TimeKey', 'Seasonal Factor'], inplace=True)
        df['Time'] = df['Time_x']
        df.drop(columns=['Time_x'], inplace=True)
        df.set_index('Time', inplace=True)
        adjusted_simulations.append(df)

    return adjusted_simulations


4. Normalisieren auf Jahresverbrauch

In [105]:
#Funktion, die Jahreslastprofil auf Jahresverbrauch skaliert
def scale_load_profile_to_annual_consumption(df: list, annual_consumption: float) -> pd.DataFrame:
    """
    Skaliert das Lastprofil auf den gewünschten Jahresverbrauch.

    Args:
        df (list): Liste von DataFrames mit stündlichem Lastprofil.
        annual_consumption (float): Vergangener Jahresverbrauch in kWh.

    Returns:
        list: Liste von DataFrames mit skaliertem Lastprofil.
    """
    scaled_profiles = []

    for profile in df:
        profile = profile.copy()
        total_load = profile['Load'].sum()
        print(f"Total Load: {total_load} kWh")
        scaling_factor = annual_consumption / total_load
        profile['Load'] *= scaling_factor
        scaled_profiles.append(profile)

    return scaled_profiles

TEST

In [83]:
#Anwendung der Funktion mit df_random_two_week_customer_load_2020
weekday_profiles = build_weekday_profile_dict(df_random_two_week_customer_load_2020, timestamp_col='Time', load_col='Load')

#Erzeuge zufälliges Jahreslastprofil für 2020
df_yearly_profile_2020 = generate_random_yearly_profile(weekday_profiles, year=2020)

df_yearly_profile_2020

#Monte Carlo
simulations = monte_carlo_simulation_load_profiles(weekday_profiles,2020, 100)

simulations


[                         Load
 2020-01-01 00:00:00  0.285668
 2020-01-01 01:00:00  1.003764
 2020-01-01 02:00:00  1.988927
 2020-01-01 03:00:00  0.479238
 2020-01-01 04:00:00  1.274651
 ...                       ...
 2020-12-31 19:00:00  1.019409
 2020-12-31 20:00:00  0.356357
 2020-12-31 21:00:00  0.013685
 2020-12-31 22:00:00  1.833496
 2020-12-31 23:00:00  1.188245
 
 [8784 rows x 1 columns],
                          Load
 2020-01-01 00:00:00  0.285668
 2020-01-01 01:00:00  1.003764
 2020-01-01 02:00:00  1.988927
 2020-01-01 03:00:00  0.479238
 2020-01-01 04:00:00  1.274651
 ...                       ...
 2020-12-31 19:00:00  1.019409
 2020-12-31 20:00:00  0.356357
 2020-12-31 21:00:00  0.013685
 2020-12-31 22:00:00  1.833496
 2020-12-31 23:00:00  1.188245
 
 [8784 rows x 1 columns],
                          Load
 2020-01-01 00:00:00  0.285668
 2020-01-01 01:00:00  1.003764
 2020-01-01 02:00:00  1.988927
 2020-01-01 03:00:00  0.479238
 2020-01-01 04:00:00  1.274651
 ...          

In [81]:
#Summiere einzelne Lastprofil Jahresverbauch auf

seasonal_factor_month = calculate_seasonal_factor_per_x(df_standard_load_profile_2019, x="Month")
seasonal_factor_month

,Time,Seasonal Factor
0,1,1.177078
1,2,1.116755
2,3,0.979929
3,4,0.923508
4,5,0.890456
5,6,0.824066
6,7,0.773264
7,8,0.846042
8,9,0.913450
9,10,1.116269


In [ ]:
adjusted_load = apply_seasonal_factors(simulations, seasonal_factor_month, by="Month")

y = scale_load_profile_to_annual_consumption(adjusted_load, 3000)

    

Total Load: 8793.238761446186 kWh
Total Load: 8816.683375197023 kWh
Total Load: 8850.222324778097 kWh
Total Load: 8807.057617223125 kWh
Total Load: 8798.046852835016 kWh
Total Load: 8836.855102552128 kWh
Total Load: 8802.429512142091 kWh
Total Load: 8846.449031269078 kWh
Total Load: 8769.54842726721 kWh
Total Load: 8793.53272329657 kWh
Total Load: 8797.779668035091 kWh
Total Load: 8813.012343042448 kWh
Total Load: 8822.1192217669 kWh
Total Load: 8825.883070380045 kWh
Total Load: 8841.123271781726 kWh
Total Load: 8857.889119128378 kWh
Total Load: 8843.888431380266 kWh
Total Load: 8832.984708805589 kWh
Total Load: 8839.47966674013 kWh
Total Load: 8839.979030189943 kWh
Total Load: 8863.211953550232 kWh
Total Load: 8881.495777606555 kWh
Total Load: 8840.856326478819 kWh
Total Load: 8800.509185730389 kWh
Total Load: 8865.962399564529 kWh
Total Load: 8901.511174759116 kWh
Total Load: 8744.07094146667 kWh
Total Load: 8846.355927973611 kWh
Total Load: 8892.414086637556 kWh
Total Load: 8799.397